# 04. Optimización de Hiperparámetros y Regularización (Workshop S5)
**Proyecto Integrador en Inteligencia Artificial (MIAR0545) — UEES**  
**Autores:** Luis Gabriel Salgado Santamaría & Daniel Fernando Salgado Santamaría  
**Fecha:** Octubre 2026 | Versión 2.0

---
### Objetivos del Notebook:
1. Diseñar la arquitectura compacta `RegularizedSmallMLP` (12,580 parámetros, reducción del 94.7%).
2. Integrar Batch Normalization y esquema de Dropout decreciente adaptativo ($0.35 ightarrow 0.245 ightarrow 0.14$).
3. Aplicar penalización $L_2$ (`weight_decay` = $10^{-3}$) y Early Stopping con paciencia de 20 épocas.
4. Comparar cuantitativamente el desempeño antes y después de la optimización.


In [1]:
import sys
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.architecture import RegularizedSmallMLP
print("Arquitectura RegularizedSmallMLP cargada correctamente.")


Arquitectura RegularizedSmallMLP cargada correctamente.


In [2]:
# Instanciación y Conteo de Parámetros de la Red Optimizada
opt_model = RegularizedSmallMLP(input_dim=12, num_classes=4, dropout_rate=0.35)
n_params = opt_model.count_parameters()
print(f"Topología: {opt_model}")
print(f"Parámetros entrenables: {n_params:,} (18.8x menor que el modelo base).")


Topología: RegularizedSmallMLP(
  (network): Sequential(
    (0): Linear(in_features=12, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.24499999999999997, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.13999999999999999, inplace=False)
    (12): Linear(in_features=32, out_features=4, bias=True)
  )
)
Parámetros entrenables: 12,580 (18.8x menor que el modelo base).


In [3]:
# Implementación de Early Stopping con Restauración de Mejor Checkpoint
class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-5):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_weights = None
        self.stopped_epoch = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

early_stopper = EarlyStopping(patience=20)
print("EarlyStopping configurado con paciencia de 20 épocas.")


EarlyStopping configurado con paciencia de 20 épocas.


In [4]:
# Carga de datos normalizados
import joblib
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "squat_features.csv")
target_col = 'label' if 'label' in df.columns else 'class_id'
from src.kinematics.vector_engine import FEATURE_NAMES

X = df[FEATURE_NAMES].values
y = df[target_col].values

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

scaler = joblib.load(PROJECT_ROOT / "models" / "scaler.pkl")
X_train_s = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_s = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_s = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_s, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_s, y_val_t), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_s, y_test_t), batch_size=32, shuffle=False)


In [5]:
# Entrenamiento Optimizado con AdamW y Regularización L2
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(opt_model.parameters(), lr=1e-3, weight_decay=1e-3)

history_opt = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, 151):
    opt_model.train()
    t_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = opt_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * len(by)
        correct += (out.argmax(dim=1) == by).sum().item()
        total += len(by)
    tr_loss = t_loss / total
    tr_acc = correct / total * 100.0

    opt_model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            out = opt_model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * len(by)
            v_corr += (out.argmax(dim=1) == by).sum().item()
            v_tot += len(by)
    va_loss = v_loss / v_tot
    va_acc = v_corr / v_tot * 100.0

    history_opt['train_loss'].append(tr_loss)
    history_opt['val_loss'].append(va_loss)
    history_opt['train_acc'].append(tr_acc)
    history_opt['val_acc'].append(va_acc)

    if early_stopper.step(va_loss, opt_model):
        print(f"Early Stopping activado en época {epoch}. Mejor val_loss: {early_stopper.best_loss:.4f}")
        opt_model.load_state_dict(early_stopper.best_weights)
        break

    if epoch % 10 == 0:
        print(f"Época {epoch:03d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} (Acc: {va_acc:.1f}%)")


Época 010 | Train Loss: 0.1018 | Val Loss: 0.0463 (Acc: 98.7%)


Época 020 | Train Loss: 0.0652 | Val Loss: 0.0242 (Acc: 99.3%)


Época 030 | Train Loss: 0.0532 | Val Loss: 0.0259 (Acc: 99.0%)


Época 040 | Train Loss: 0.0431 | Val Loss: 0.0175 (Acc: 99.2%)


Época 050 | Train Loss: 0.0392 | Val Loss: 0.0133 (Acc: 99.5%)


Época 060 | Train Loss: 0.0376 | Val Loss: 0.0165 (Acc: 99.3%)


Época 070 | Train Loss: 0.0318 | Val Loss: 0.0100 (Acc: 99.7%)


Época 080 | Train Loss: 0.0353 | Val Loss: 0.0099 (Acc: 99.8%)


Early Stopping activado en época 87. Mejor val_loss: 0.0095


In [6]:
# Evaluación Final en Test Set Ciego del Modelo Regularizado
opt_model.eval()
test_loss, t_corr, t_tot = 0.0, 0, 0
with torch.no_grad():
    for bx, by in test_loader:
        out = opt_model(bx)
        test_loss += criterion(out, by).item() * len(by)
        t_corr += (out.argmax(dim=1) == by).sum().item()
        t_tot += len(by)

print(f"=== RESULTADOS TEST SET CIEGO (REGULARIZED SMALL MLP) ===")
print(f"Test Loss: {test_loss / t_tot:.4f} (vs 3.5206 en Modelo Base)")
print(f"Test Accuracy: {t_corr / t_tot * 100:.2f}% (vs 85.00% en Modelo Base)")
print(f"Generalization Gap: {abs(history_opt['val_loss'][-1] - history_opt['train_loss'][-1]):.4f}")


=== RESULTADOS TEST SET CIEGO (REGULARIZED SMALL MLP) ===
Test Loss: 0.0072 (vs 3.5206 en Modelo Base)
Test Accuracy: 99.88% (vs 85.00% en Modelo Base)
Generalization Gap: 0.0173


In [7]:
# Guardado del Modelo Optimizado
torch.save(opt_model.state_dict(), PROJECT_ROOT / "models" / "best_model.pth")
print("[OK] Modelo RegularizedSmallMLP guardado con éxito en models/best_model.pth")


[OK] Modelo RegularizedSmallMLP guardado con éxito en models/best_model.pth


### Resumen de Mejoras Cuantitativas:
* **Pérdida en Test Set:** Reducida de $3.5206 ightarrow 0.4483$ (**-87.3% de error**).
* **Tamaño del Modelo:** Reducido de 237,188 a 12,580 parámetros (**18.8× más compacto**).
* **Brecha de Generalización:** Prácticamente nula ($+0.0031$), eliminando cualquier síntoma de sobreajuste.
